In [59]:
import asyncio
import random
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from typing import Literal, Optional
from pydantic import BaseModel, Field

from openai import AsyncOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
chunks = pd.read_parquet('../data/chunked_conclusions_557k_2026-01-26.parquet')
chunks = chunks.set_index(['openalex_id', 'chunk_idx'])['text']

In [6]:
df = pd.read_parquet('../results_557k/cluster_impacts_2026-03-12.parquet')

In [7]:
clusters = pd.read_csv('../clustering/cluster_representatives_2026-03-10.csv')
clusters = clusters.set_index('cluster_id')['text']

In [9]:
def cluster_impacts_summary(cluster_df):
    s = ''
    for cat, subgroup in cluster_df.groupby('impact_category'):
        s += cat
        for row in subgroup.itertuples():
            s += f"""
- {row.impact_dim}
  - positive: {row.positive}
  - negative: {row.negative}
  - neutral: {row.neutral}

"""
    return s.strip()

def cluster_impacts_context(cluster_df, max_samples_per_dim=2):
    s = ''
    for cat, subgroup in cluster_df.groupby('impact_category'):
        for row in subgroup.itertuples():
            s += f'{cat} - {row.impact_dim}:\n\n'
            for direction in ['positive', 'negative', 'neutral']:
                refs = getattr(row, f'{direction}_refs').tolist()
                random.shuffle(refs)
                for ref in refs[:max_samples_per_dim]:
                    oaid, chunk_idx = ref.split('_')
                    s += f"{direction.upper()} example:\n"
                    s += f'> {chunks.loc[(oaid, int(chunk_idx))].values[0]}\n\n'
        s += '-'*80 + '\n\n'
    return s.strip()


def build_cluster_context(cluster_id, max_samples_per_dim=2):
    cluster_policy = clusters.loc[cluster_id]
    s = f"Cluster {cluster_id} - Representative policy:\n\n{cluster_policy}\n\n"
    cluster_df = df[df['cluster_id'] == cluster_id]
    summary = cluster_impacts_summary(cluster_df)
    s += f"Impacts assessment summary :\n\n{summary}\n\n"
    context = cluster_impacts_context(cluster_df, max_samples_per_dim)
    s += f"Impacts assessment context samples:\n\n{context}\n\n"
    return s

In [13]:
d = {}
with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(build_cluster_context, cluster_id): cluster_id for cluster_id in df['cluster_id'].unique()}
    for future in tqdm(as_completed(futures), total=len(futures)):
        cluster_id = futures[future]
        d[int(cluster_id)] = future.result()

/tmp/ipykernel_67996/541382866.py:26: PerformanceWarning: indexing past lexsort depth may impact performance.
  s += f'> {chunks.loc[(oaid, int(chunk_idx))].values[0]}\n\n'


  0%|          | 0/2481 [00:00<?, ?it/s]

In [20]:
sufficiency_eval_prompt = """
Sufficiency is a set of policy measures and daily practices which avoid the demand for energy, materials, land, water, and other natural resources, while delivering wellbeing for all within planetary boundaries.
Importantly, sufficiency isn't efficiency, which is doing more or the same with less.
Sufficiency is about *avoiding* demand. Also sufficiency entails both a physical ceiling and a social floor.

The following impact dimensions should be considered when evaluating whether a policy is a sufficiency policy or not:
- natural resources: freshwater, marine resources, wetlands, metals and ores, non-metallic minerals, fossil fuels, agricultural land, forests, urban land, biomass.
- wellbeing: housing, jobs, education, civic engagement, life satisfaction, work-life balance, income, community, environment, health, safety.
- justice: distributional, procedural, corrective, recognitional, transitional.
- planetary boundaries: land system change, climate change, biosphere integrity, biogeochemical flows, ocean acidification, freshwater use, atmospheric aerosol loading, ozone depletion, introduction of novel entities.

You will find below a policy along with retrieved evidences of its impacts on the above dimensions.
You are first shown a summary of evidence for each impact, where e.g. "positive: 10" means that 10 references mention a positive/beneficial impact of the policy on the given dimension.
Then, you are shown a few examples of retrieved evidences for each impact dimension and direction (positive, negative, neutral).

Stage 1 : classify the policy between the following categories:
- DECARBONATION: supply-side energy decarbonation (renewables, nuclear...)
- EFFICIENCY: efficiency-based policy (optimization, energy intensity reduction...)
- SUFFICIENCY-COMPATIBLE: avoiding upfront the demand for resources
- NOT-COMPATIBLE: no or negative impact on resource use

Stage 2 : if the policy is SUFFICIENCY-COMPATIBLE, classify the policy into one of the below three categories:
- Sufficiency (S): Primarily and directly contributes to human well-being while significantly reducing resource demand and staying within planetary boundaries.
- Potential Sufficiency (PS): Has the potential to achieve sufficiency, but requires explicit, specific, and significant transformation (e.g., equity corrections, policy integration) to overcome limitations and align with all boundaries.
- Not Sufficiency (NS): The policy's effect on human needs is indirect, often resulting in breaches of planetary limits or social foundations. The policy might turn up to be a violator of basic needs.  Or intrinsically undermines one or both boundaries (social foundations or planetary limits).

**Important rule**: use only contextual information, not general knowledge or assumptions about the policy.
"""


def build_messages(impact_summary: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": sufficiency_eval_prompt},
        {"role": "user", "content": f"Policy impact summary : \n\n {impact_summary}"}
    ]


class SufficiencyClassificationResponse(BaseModel):
    stage1_reasoning: str = Field(description="Short reasoning justifying the stage 1 classification based on the provided evidence.")
    stage1_classification: Literal['DECARBONATION', 'EFFICIENCY', 'SUFFICIENCY-COMPATIBLE', 'NOT-COMPATIBLE']
    stage2_reasoning: str = Field(description="Short reasoning justifying the stage 2 classification based on the provided evidence. Should be empty if stage1_classification is not 'SUFFICIENCY-COMPATIBLE'.")
    stage2_classification: Optional[Literal['S', 'PS', 'NS']] = Field(description="If and only if stage 1 classification is SUFFICIENCY-COMPATIBLE, stage 2 classification of the policy's sufficiency level.")

In [58]:
client = AsyncOpenAI(
    base_url=os.getenv("GENERATION_API_URL"),
    api_key=os.getenv("SCW_SECRET_KEY")
)

In [73]:
async def classify_sufficiency(evidences_summary: str, model = "qwen3-235b-a22b-instruct-2507"):
    try:
        kwargs = {
            "model": model,
            "messages": build_messages(evidences_summary),
            "max_tokens": 1024,
            "temperature": 0.01,
            "top_p": 0.1,
            "timeout": 120,
            "reasoning_effort": "low",
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": SufficiencyClassificationResponse.__name__,
                    "schema": SufficiencyClassificationResponse.model_json_schema(),
                },
            },
        }

        response = await client.chat.completions.create(**kwargs)
        return SufficiencyClassificationResponse.model_validate_json(
            response.choices[0].message.content
        )
    except Exception as e:
        print(f"Error generating response: {e}")
        return e

async def batch_classify_sufficiency(
    batch: pd.Series,
):
    results = await asyncio.gather(*[classify_sufficiency(row) for _, row in batch.items()])
    return results

In [54]:
classify_sufficiency(d[1])

SufficiencyClassificationResponse(stage1_reasoning="The policy 'Preserving natural lands for flood management' directly avoids the demand for engineered flood control infrastructure (e.g., levees, dams, drainage systems) by relying on natural systems such as wetlands, forests, and floodplains to absorb and regulate floodwaters. This constitutes demand avoidance for energy, materials, and land typically required for built infrastructure. The policy aligns with the core principle of sufficiency by maintaining natural land uses instead of converting them to urban or industrial uses, thereby preventing increased resource consumption. Evidence shows positive impacts across multiple natural resources (wetlands, forests, freshwater, agricultural land) and planetary boundaries (biosphere integrity, land system change, climate change), confirming its role in reducing pressure on ecosystems and staying within ecological limits.", stage1_classification='SUFFICIENCY-COMPATIBLE', stage2_reasoning="

In [55]:
_.model_dump()

{'stage1_reasoning': "The policy 'Preserving natural lands for flood management' directly avoids the demand for engineered flood control infrastructure (e.g., levees, dams, drainage systems) by relying on natural systems such as wetlands, forests, and floodplains to absorb and regulate floodwaters. This constitutes demand avoidance for energy, materials, and land typically required for built infrastructure. The policy aligns with the core principle of sufficiency by maintaining natural land uses instead of converting them to urban or industrial uses, thereby preventing increased resource consumption. Evidence shows positive impacts across multiple natural resources (wetlands, forests, freshwater, agricultural land) and planetary boundaries (biosphere integrity, land system change, climate change), confirming its role in reducing pressure on ecosystems and staying within ecological limits.",
 'stage1_classification': 'SUFFICIENCY-COMPATIBLE',
 'stage2_reasoning': "The policy is classifi

In [63]:
imp = pd.Series(d)

In [65]:
import time

In [74]:
# process by batch
batch_size = 20
rpm_quota = 600
wait_time = 60 / (rpm_quota / batch_size)
results = []
for i in tqdm(range(0, len(imp), batch_size)):
    batch = imp.iloc[i:i+batch_size]
    preds = await batch_classify_sufficiency(batch)
    results.extend(preds)
    time.sleep(0.1)

/tmp/ipykernel_67996/2382894588.py:5: RuntimeWarning: coroutine 'AsyncCompletions.create' was never awaited
  results = []


  0%|          | 0/125 [00:00<?, ?it/s]

/tmp/ipykernel_67996/2382894588.py:8: RuntimeWarning: coroutine 'AsyncCompletions.create' was never awaited
  preds = await batch_classify_sufficiency(batch)
/home/francois/projects/13_democratiser_sobriete/policy_analysis/.venv/lib/python3.12/site-packages/pydantic/json_schema.py:516: RuntimeWarning: coroutine 'AsyncCompletions.create' was never awaited
  current_handler = _schema_generation_shared.GenerateJsonSchemaHandler(self, handler_func)


In [78]:
imp

2       Cluster 2 - Representative policy:\n\nContinuo...
12      Cluster 12 - Representative policy:\n\nAnalysi...
7       Cluster 7 - Representative policy:\n\nCollecti...
0       Cluster 0 - Representative policy:\n\nInsuranc...
10      Cluster 10 - Representative policy:\n\nEstabli...
                              ...                        
3340    Cluster 3340 - Representative policy:\n\nAlloc...
3759    Cluster 3759 - Representative policy:\n\nImple...
3702    Cluster 3702 - Representative policy:\n\nInves...
3395    Cluster 3395 - Representative policy:\n\nRobus...
3485    Cluster 3485 - Representative policy:\n\nEncou...
Length: 2481, dtype: str

In [79]:
resdf = pd.DataFrame([r.model_dump() if isinstance(r, SufficiencyClassificationResponse) else {"error": str(r)} for r in results])
resdf

,stage1_reasoning,stage1_classification,stage2_reasoning,stage2_classification
0,The policy involves continuous monitoring of f...,SUFFICIENCY-COMPATIBLE,The policy is classified as Sufficiency (S) be...,S
1,The policy 'Analysis of consumer perceptions a...,NOT-COMPATIBLE,The policy does not primarily or directly redu...,NS
2,The policy focuses on collecting social and ec...,SUFFICIENCY-COMPATIBLE,The policy has strong potential for sufficienc...,PS
3,The policy of insurance programs and tax incen...,SUFFICIENCY-COMPATIBLE,The policy is classified as Potential Sufficie...,PS
4,The policy—establishment of online learning pl...,SUFFICIENCY-COMPATIBLE,The policy is classified as 'Sufficiency (S)' ...,S
...,...,...,...,...
2476,The policy involves allocating agricultural la...,SUFFICIENCY-COMPATIBLE,The policy has strong potential for sufficienc...,PS
2477,The policy focuses on implementing tailored su...,SUFFICIENCY-COMPATIBLE,The policy directly contributes to human well-...,S
2478,The policy centers on investments in infrastru...,DECARBONATION,Although the policy reduces net emissions and ...,NS
2479,"The policy focuses on building robust, context...",SUFFICIENCY-COMPATIBLE,The policy is classified as Sufficiency (S) be...,S


In [87]:
final.columns

Index([                'index',                       0,
            'stage1_reasoning', 'stage1_classification',
            'stage2_reasoning', 'stage2_classification'],
      dtype='object')

In [89]:
final = pd.concat((pd.DataFrame(imp).reset_index(), resdf), axis=1)
final = final.rename(columns={'index': 'cluster_id', 0: 'context'})

In [90]:
final.stage1_classification.value_counts()

stage1_classification
SUFFICIENCY-COMPATIBLE    1848
EFFICIENCY                 280
NOT-COMPATIBLE             218
DECARBONATION              135
Name: count, dtype: int64

In [91]:
final.stage2_classification.value_counts()

stage2_classification
S     1327
NS     582
PS     522
Name: count, dtype: int64

In [94]:
final = final.sort_values('cluster_id')

In [95]:
final

,cluster_id,context,stage1_reasoning,stage1_classification,stage2_reasoning,stage2_classification
3,0,Cluster 0 - Representative policy:\n\nInsuranc...,The policy of insurance programs and tax incen...,SUFFICIENCY-COMPATIBLE,The policy is classified as Potential Sufficie...,PS
10,1,Cluster 1 - Representative policy:\n\nPreservi...,The policy 'Preserving natural lands for flood...,SUFFICIENCY-COMPATIBLE,The policy is classified as 'Sufficiency (S)' ...,S
0,2,Cluster 2 - Representative policy:\n\nContinuo...,The policy involves continuous monitoring of f...,SUFFICIENCY-COMPATIBLE,The policy is classified as Sufficiency (S) be...,S
12,3,Cluster 3 - Representative policy:\n\nIncreasi...,The policy of increasing restoration funding i...,SUFFICIENCY-COMPATIBLE,The policy of increasing restoration funding p...,S
15,4,Cluster 4 - Representative policy:\n\nImplemen...,The policy focuses on mitigating flood risks i...,SUFFICIENCY-COMPATIBLE,The policy primarily and directly contributes ...,S
...,...,...,...,...,...,...
2479,3395,Cluster 3395 - Representative policy:\n\nRobus...,"The policy focuses on building robust, context...",SUFFICIENCY-COMPATIBLE,The policy is classified as Sufficiency (S) be...,S
2480,3485,Cluster 3485 - Representative policy:\n\nEncou...,The policy 'Encouraging community engagement a...,SUFFICIENCY-COMPATIBLE,The policy is classified as Potential Sufficie...,PS
2478,3702,Cluster 3702 - Representative policy:\n\nInves...,The policy centers on investments in infrastru...,DECARBONATION,Although the policy reduces net emissions and ...,NS
2474,3708,Cluster 3708 - Representative policy:\n\nTarge...,The policy focuses on reducing deterioration o...,SUFFICIENCY-COMPATIBLE,The policy has the potential to significantly ...,PS


In [97]:
clusters = pd.read_csv('../clustering/cluster_representatives_2026-03-10.csv')

In [101]:
thistimeitsfinal = clusters.merge(final, on='cluster_id', how='left')
thistimeitsfinal

,cluster_id,text,count,context,stage1_reasoning,stage1_classification,stage2_reasoning,stage2_classification
0,0,Insurance programs and tax incentives for floo...,715,Cluster 0 - Representative policy:\n\nInsuranc...,The policy of insurance programs and tax incen...,SUFFICIENCY-COMPATIBLE,The policy is classified as Potential Sufficie...,PS
1,1,Preserving natural lands for flood management,447,Cluster 1 - Representative policy:\n\nPreservi...,The policy 'Preserving natural lands for flood...,SUFFICIENCY-COMPATIBLE,The policy is classified as 'Sufficiency (S)' ...,S
2,2,Continuous monitoring of fishways to optimise ...,1536,Cluster 2 - Representative policy:\n\nContinuo...,The policy involves continuous monitoring of f...,SUFFICIENCY-COMPATIBLE,The policy is classified as Sufficiency (S) be...,S
3,3,Increasing restoration funding,79,Cluster 3 - Representative policy:\n\nIncreasi...,The policy of increasing restoration funding i...,SUFFICIENCY-COMPATIBLE,The policy of increasing restoration funding p...,S
4,4,Implementation of measures to mitigate the ris...,1031,Cluster 4 - Representative policy:\n\nImplemen...,The policy focuses on mitigating flood risks i...,SUFFICIENCY-COMPATIBLE,The policy primarily and directly contributes ...,S
...,...,...,...,...,...,...,...,...
2620,3673,Normativas de gestión de riesgos hídricos para...,3,NaN,NaN,NaN,NaN,NaN
2621,3702,Investments in infrastructure directly address...,4,Cluster 3702 - Representative policy:\n\nInves...,The policy centers on investments in infrastru...,DECARBONATION,Although the policy reduces net emissions and ...,NS
2622,3708,Target Environment-Specific Deterioration Mana...,6,Cluster 3708 - Representative policy:\n\nTarge...,The policy focuses on reducing deterioration o...,SUFFICIENCY-COMPATIBLE,The policy has the potential to significantly ...,PS
2623,3710,Equity-led policies,3,NaN,NaN,NaN,NaN,NaN


In [102]:
thistimeitsfinal.to_csv('cluster_sufficiency_classification_2026-03-19.csv', index=False)